In [1]:
import pandas as pd
import numpy as np

def clean_power_data(input_file, output_file):
    print(f"Đang kiểm tra file '{input_file}'...")
    
    try:
        df = pd.read_csv(input_file, sep=',', low_memory=False, na_values=['?'])
        
        if df.shape[1] <= 1:
            print("Phát hiện dùng sai dấu phân cách, đang thử lại với dấu ';'...")
            df = pd.read_csv(input_file, sep=';', low_memory=False, na_values=['?'])

        print("Đang định dạng ngày tháng...")
        df['dt'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], dayfirst=True)
        df.set_index('dt', inplace=True)
        
        df = df[['Global_active_power']]
        
        df['Global_active_power'] = pd.to_numeric(df['Global_active_power'], errors='coerce')

        print("Đang xử lý giá trị thiếu (NaN)...")
        df = df.ffill()

        print("Đang nén dữ liệu theo Ngày (Daily Resample)...")
        df_daily = df['Global_active_power'].resample('D').mean()
    
        df_daily.to_csv(output_file)
        print(f"--- THÀNH CÔNG! Đã tạo file '{output_file}' ---")

    except Exception as e:
        print(f"Lỗi rồi bạn ơi: {e}")
        print("Mẹo: Hãy kiểm tra xem file 'điện.csv' có đang mở bằng Excel không? Nếu có thì đóng lại nhé!")
clean_power_data('điện.csv', 'power_daily_clean.csv')

Đang kiểm tra file 'điện.csv'...
Lỗi rồi bạn ơi: [Errno 2] No such file or directory: 'điện.csv'
Mẹo: Hãy kiểm tra xem file 'điện.csv' có đang mở bằng Excel không? Nếu có thì đóng lại nhé!


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

In [3]:
df = pd.read_csv('power_daily_clean.csv', index_col='dt', parse_dates=True)
data = df.values

In [4]:
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)

In [5]:
train_len = int(len(scaled_data) * 0.8)
train_data = scaled_data[:train_len]

In [7]:
def create_dataset(dataset, look_back=30):
    X, Y = [], []
    for i in range(look_back, len(dataset)):
        X.append(dataset[i-look_back:i, 0])
        Y.append(dataset[i, 0])
    return np.array(X), np.array(Y)

look_back = 30
X_train, y_train = create_dataset(train_data, look_back)
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))

In [8]:
model = Sequential([
    LSTM(units=64, return_sequences=True, input_shape=(X_train.shape[1], 1)),
    Dropout(0.2),
    
    LSTM(units=32, return_sequences=False),
    Dropout(0.2),
    
    Dense(units=1) 
])

model.compile(optimizer='adam', loss='mean_squared_error')

c:\Users\admin\Desktop\AI\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [9]:
print("Đang cho LSTM học thói quen dùng điện...")
model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=1)

Đang cho LSTM học thói quen dùng điện...
Epoch 1/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0195
Epoch 2/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0131
Epoch 3/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0120
Epoch 4/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0126
Epoch 5/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0122
Epoch 6/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0115
Epoch 7/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0113
Epoch 8/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0116
Epoch 9/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0112
Epoch 10/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0115


In [10]:
test_data = scaled_data[train_len - look_back:]
X_test, y_test = create_dataset(test_data, look_back)
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))

predictions = model.predict(X_test)
predictions = scaler.inverse_transform(predictions)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step


In [13]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Dự báo train
train_predict_lstm = model.predict(X_train)
train_predict_lstm = scaler.inverse_transform(train_predict_lstm)
y_train_unscaled = scaler.inverse_transform(y_train.reshape(-1, 1))
y_test_unscaled_lstm = scaler.inverse_transform(y_test.reshape(-1, 1))
predictions_lstm = scaler.inverse_transform(predictions_raw)  # predictions chưa inverse

# Metrics
rmse = np.sqrt(mean_squared_error(y_test_unscaled_lstm, predictions_lstm))
mae = mean_absolute_error(y_test_unscaled_lstm, predictions_lstm)
r2 = r2_score(y_test_unscaled_lstm, predictions_lstm)
mape = np.mean(np.abs((y_test_unscaled_lstm - predictions_lstm) / y_test_unscaled_lstm)) * 100

plt.style.use('default')
plt.figure(figsize=(15, 6))

# Actual
plt.plot(range(len(y_train_unscaled) + len(y_test_unscaled_lstm)),
         np.concatenate([y_train_unscaled, y_test_unscaled_lstm]),
         color='steelblue', linewidth=1.2, label='Actual')

# Train Predict
plt.plot(range(len(train_predict_lstm)),
         train_predict_lstm,
         color='orange', linewidth=1.2, linestyle='--', label='Train Predict')

# Test Predict
plt.plot(range(len(y_train_unscaled), len(y_train_unscaled) + len(predictions_lstm)),
         predictions_lstm,
         color='green', linewidth=1.2, linestyle='--', label='Test Predict')

plt.title(f'LSTM Prediction\nRMSE: {rmse:.2f} | MAE: {mae:.2f} | R2: {r2:.4f} | MAPE: {mape:.2f}%')
plt.xlabel('Time')
plt.ylabel('Công suất (Global_active_power)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


NameError: name 'predictions_raw' is not defined

In [14]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 64)         │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 88,037 (343.90 KB)

 Trainable params: 29,345 (114.63 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 58,692 (229.27 KB)